In [8]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DecimalType, TimestampType
from src.spark_session import get_spark
from src.config import ORDER_ITEMS_SILVER_PATH, ORDER_ITEMS_RAW_PATH

In [2]:
spark = get_spark("SilverOrderItems")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/06 11:50:20 WARN Utils: Your hostname, Branimirs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.199 instead (on interface en0)
26/08/06 11:50:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/branimiranastasov/PycharmProjects/azure_retail_lakehouse/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/06 11:50:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applic

In [3]:
order_items_raw_df = spark.read.option("header", True).option("inferSchema", False).csv(str(ORDER_ITEMS_RAW_PATH))
order_items_raw_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: string (nullable = true)
 |-- price: string (nullable = true)
 |-- freight_value: string (nullable = true)



In [4]:
order_items_raw_df.show(10, truncate=False)

+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+------+-------------+
|order_id                        |order_item_id|product_id                      |seller_id                       |shipping_limit_date|price |freight_value|
+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+------+-------------+
|00010242fe8c5a6d1ba2dd792cb16214|1            |4244733e06e7ecb4970a6e2683c13e61|48436dade18ac8b2bce089ec2a041202|2017-09-19 09:45:35|58.90 |13.29        |
|00018f77f2f0320c557190d7a144bdd3|1            |e5f2d52b802189ee658865ca93d83a8f|dd7ddc04e1b6c2c614352b383efe2d36|2017-05-03 11:05:13|239.90|19.93        |
|000229ec398224ef6ca0657da4fc703e|1            |c777355d18b72b67abbeef9df44fd0fd|5b51032eddd242adc84c38acab88f23d|2018-01-18 14:48:30|199.00|17.87        |
|00024acbcdf0a6daa1e931b038114c75|1            |7634da152a4610f1

In [5]:
unique_order_item_ids = order_items_raw_df.select("order_item_id").distinct().count()
print("Rows: ", order_items_raw_df.count())
print("Unique order item ids: ", unique_order_item_ids)
print("Unique product_id: ", order_items_raw_df.select("product_id").distinct().count())

Rows:  112650
Unique order item ids:  21
Unique product_id:  32951


In [6]:
distinct_item_keys = (
    order_items_raw_df
    .select("order_id", "order_item_id")
    .distinct()
    .count()
)

total_rows = order_items_raw_df.count()

print("Total rows: ", total_rows)
print("Distinct order items keys:", distinct_item_keys)
print("Combined key is unique:", total_rows == distinct_item_keys)

Total rows:  112650
Distinct order items keys: 112650
Combined key is unique: True


In [9]:
orders_items_schema = StructType([
    StructField("order_id", StringType(), nullable=False),
    StructField("order_item_id", IntegerType(), nullable=False),
    StructField("product_id", StringType(), nullable=False),
    StructField("seller_id", StringType(), nullable=False),
    StructField("shipping_limit_date", TimestampType(), nullable=True),
    StructField("price", DecimalType(12, 2), nullable=True),
    StructField("freight_value", DecimalType(12, 2), nullable=True),
])

In [11]:
orders_items_typed_df = spark.read.option("header", True).schema(orders_items_schema).csv(str(ORDER_ITEMS_RAW_PATH))
orders_items_typed_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: decimal(12,2) (nullable = true)
 |-- freight_value: decimal(12,2) (nullable = true)



In [12]:
order_items_clean_df = (
    orders_items_typed_df
    .filter(F.col("order_id").isNotNull())
    .filter(F.col("order_item_id").isNotNull())
    .filter(F.col("product_id").isNotNull())
    .withColumn(
        "item_total",
        F.col("price") + F.col("freight_value")
    )
)

order_items_clean_df.show(5, truncate=False)

+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+------+-------------+----------+
|order_id                        |order_item_id|product_id                      |seller_id                       |shipping_limit_date|price |freight_value|item_total|
+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+------+-------------+----------+
|00010242fe8c5a6d1ba2dd792cb16214|1            |4244733e06e7ecb4970a6e2683c13e61|48436dade18ac8b2bce089ec2a041202|2017-09-19 09:45:35|58.90 |13.29        |72.19     |
|00018f77f2f0320c557190d7a144bdd3|1            |e5f2d52b802189ee658865ca93d83a8f|dd7ddc04e1b6c2c614352b383efe2d36|2017-05-03 11:05:13|239.90|19.93        |259.83    |
|000229ec398224ef6ca0657da4fc703e|1            |c777355d18b72b67abbeef9df44fd0fd|5b51032eddd242adc84c38acab88f23d|2018-01-18 14:48:30|199.00|17.87        |216.87    

In [13]:
orders_items_typed_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: decimal(12,2) (nullable = true)
 |-- freight_value: decimal(12,2) (nullable = true)



In [14]:
order_items_clean_df.coalesce(2).write.mode("overwrite").parquet(str(ORDER_ITEMS_SILVER_PATH))